In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train_final.csv")
test = pd.read_csv("../data/processed/test_final.csv")

X_train = train.drop("habitable", axis=1)
y_train = train["habitable"]

X_test = test.drop("habitable", axis=1)
y_test = test["habitable"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1487, 6)
Test shape: (372, 6)


In [2]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

class_weight_dict

{np.int64(0): np.float64(0.5027045300878973), np.int64(1): np.float64(92.9375)}

In [3]:
X_train.isna().sum()

log_pl_orbsmax      0
log_st_lum          0
log_pl_rade         0
log_teq             0
log_st_teff         1
log_stellar_flux    0
dtype: int64

In [4]:
# Find rows with NaN
nan_rows = X_train[X_train.isna().any(axis=1)]

nan_rows

,log_pl_orbsmax,log_st_lum,log_pl_rade,log_teq,log_st_teff,log_stellar_flux
917,-0.392302,-1.022131,-1.251551,-0.374649,NaN,-0.688476


In [5]:
# Drop NaN rows
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

print("New train shape:", X_train.shape)

New train shape: (1486, 6)


In [6]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully.")



Model trained successfully.


In [7]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

cm

array([[355,  15],
       [  0,   2]])

In [8]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98       370
           1       0.12      1.00      0.21         2

    accuracy                           0.96       372
   macro avg       0.56      0.98      0.59       372
weighted avg       1.00      0.96      0.98       372



In [9]:
y_probs = model.predict_proba(X_test)[:, 1]

y_probs[:10]

array([2.01704945e-02, 1.69271295e-02, 1.13706243e-01, 7.12535082e-12,
       2.06192700e-04, 1.59644164e-07, 2.45809786e-05, 1.10097508e-11,
       1.59431657e-04, 3.14639346e-01])

In [10]:
import pandas as pd

results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "probability": y_probs
})

results[results["predicted"] == 1]

,actual,predicted,probability
11,0,1,0.539176
55,0,1,0.974554
70,0,1,0.993035
79,0,1,0.581250
98,0,1,0.824749
119,0,1,0.882843
172,0,1,0.540897
191,1,1,0.817738
240,0,1,0.982904
246,0,1,0.906984


In [11]:
results["probability"].describe()

count    3.720000e+02
mean     4.838181e-02
std      1.748085e-01
min      6.120377e-16
25%      7.260136e-09
50%      2.721005e-06
75%      1.983311e-04
max      9.930346e-01
Name: probability, dtype: float64

In [13]:
class_weight="balanced"

In [14]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_balanced.fit(X_train, y_train)

y_pred_bal = model_balanced.predict(X_test)

from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_bal)

array([[355,  15],
       [  0,   2]])